This is a reimplementation of Scott's Mathcad worksheet in a Python notebook to help understand the optical design process and to provide a tool for adjusting the parameters for the SMA upgrade.

In [1]:
import scipy as sp

In [2]:
from scipy.constants import physical_constants, c, milli
mm = milli

## Element positions

In [3]:
z_L1 =    0*mm     # Feed lens
z_M6 = 1761*mm     # Mirror 6
z_M5 = 2161*mm     # Mirror 5
z_M4 = 3456.64*mm  # Mirror 4
z_M3 = 3891.74*mm  # Mirror 3
z_pv = 5932.74*mm  # Primary vertex aperture
z_sr = 8357.74*mm  # Secondary edge plane
z_vf = -780*mm

In [4]:
z_sr - z_M3

4.465999999999999

z coordinates are measured along the optical path from receiver to secondary, with z=0 corresponding to the top surface of the 80K shield interface in the receiver inserts.

##  Receiver Beam

The receiver beam at all frequencies emerges from a virtual image of the feed located at

In [5]:
z_v = -780*mm    # Virtual image location

with a radius of curvature and beam waist radius of

In [6]:
r_v = -1291.8*mm
w_v = 30.25*mm

These values minimize the beam size through te optics up to M6 while keeping the beam size at the dewar window within reasonable bounds. The virtual image is matched to a real image at the aperture of each mixer feedhorn by a lens located at the 80K shield.

## Magnification
The secondary mirror is located at an image of the feed horn.  For the Bessel illumination produced by a scalar feed, the antenna gain is maximized with a 10.0 dB illumination taper. For the Gaussian mode set chosen to maximize the power carried by the fundamental mode, the illumination taper for the fundamental mode is 12.3 dB.

Beam waist radius for 10.0 dB edge taper:

$T_e(\mathrm{dB}) = -10 \log \big( {e^{-\frac{2R^2}{w^2}}} \big)$

$\frac{r_e}{w} = 0.3393 \times T_e(\mathrm{dB})^{0.5}$

In [7]:
Mode0_edgeTaper = 12.3 # Edge taper for fundamental mode in dB

R_sr = 175*mm           # Secondary mirror radius
a_sr = 1.306130*R_sr    # Radius of feed image for 10 dB edge taper
print(f"Radius of feed image: a_sr = {a_sr/mm:.1f} mm")
w_sr = 1.0/(0.3393*Mode0_edgeTaper**0.5)*R_sr    # Beam waist radius for 10 dB edge taper
print(f"Beam radius: w_sr = {w_sr/mm:.1f} mm")

Radius of feed image: a_sr = 228.6 mm
Beam radius: w_sr = 147.1 mm


Magnification from the secondary to virtual image is given by:

In [8]:
m = w_v / w_sr
print(f"Magnification: m = {m:.4f}")

Magnification: m = 0.2057


## Intermediate image properties

imaging conditions from T.S. Chu IEEE Trans Ant. Prop. AP-31 pp614, 1983.

### Distances
$L_1 = z_{M5} - z_v$

$L_2 = z_i - z_{M5}$

$L_3 = z_{M4} - z_i$

$L_4 = z_{sr} - z_{M4}$

In [9]:
L_1 = z_M5 - z_v
print(f"Distance from virtual feed to M5 : L_1 = {(L_1/mm):.1f} mm")
L_4 = z_sr - z_M4
print(f"Distance from M4 to secondary : L_4 = {(L_4/mm):.1f} mm")

Distance from virtual feed to M5 : L_1 = 2941.0 mm
Distance from M4 to secondary : L_4 = 4901.1 mm


### Magnification relations
$\frac{w_v}{w_i} = \frac{L_1}{L_2}$

$\frac{w_i}{w_{sr}} = \frac{L_3}{L_4}$

$m = \frac{L_1}{L_2} \frac{L_3}{L_4}$

$L_3 = z_{M4} - z_{M5} - L_2$

$m = \frac{L_1}{L_2} \frac{z_{M4}-z_{M5}-L_2}{L_4}$

$L_2 = L_1 \frac{z_{M4} - z_{M5}}{m L_4 + L_1}$

In [10]:
L_2 = L_1 * (z_M4 - z_M5)/(m*L_4 + L_1)
print(f"Distance from intermediate image to M5 : L_2 = {L_2/mm:.1f} mm")

Distance from intermediate image to M5 : L_2 = 964.9 mm


In [11]:
L_3 = z_M4 - z_M5 - L_2
print(f"Distance from intermediate image to M4 : L_3 = {L_3/mm:.1f} mm")

Distance from intermediate image to M4 : L_3 = 330.8 mm


### Properties of intermediate image

In [12]:
z_i = z_M5 + L_2
print(f"z position of intermediate image : z_i = {(z_i/mm):.1f} mm")

z position of intermediate image : z_i = 3125.9 mm


In [13]:
w_i = w_v*L_2/L_1
print(f"beam radius of intermediate image : w_i = {(w_i/mm):.4f} mm")

beam radius of intermediate image : w_i = 9.9245 mm


### Focal lengths

In [14]:
f_M5 = L_1 * L_2 / (L_2 + L_1)
print(f"focal length of M5 : f_M5 = {(f_M5/mm):.3f} mm")

focal length of M5 : f_M5 = 726.529 mm


In [15]:
f_M4 = L_3 * L_4 / (L_3 + L_4)
print(f"focal length of M4 : f_M4 = {(f_M4/mm):.3f} mm")

focal length of M4 : f_M4 = 309.841 mm


### Radius of curvature

In [16]:
r_i = 1 / ((1/L_2)*(1 + (L_1/L_2)*(1 + (L_1/r_v))))
print(f"radius of curvature of intermediate image : r_i = {(r_i/mm):.3f} mm")

radius of curvature of intermediate image : r_i = -333.721 mm


In [17]:
r_sr = 1 / ((1/L_4)*(1 + (L_3/L_4)*(1 + (L_3/r_i))))
print(f"radius of curvature of secondary image : r_sr = {(r_sr/mm):.3f} mm")

radius of curvature of secondary image : r_sr = 4898.157 mm


## Lens and feed horn properties

$L_5 = z_{L1} - z_f$

In [18]:
L_6 = z_v - z_L1
print(f"Distance from L1 to virtual image : L_6 = {(L_6/mm):.1f} mm")

Distance from L1 to virtual image : L_6 = -780.0 mm
